In [ ]:
import static_ffmpeg
static_ffmpeg.add_paths()

# Emotion Cause Pair Extraction (ECPE) with Multimodal Fusion

This notebook implements a Unified Multimodal Framework for **Emotion Cause Pair Extraction (ECPE)**. The task involves not only identifying the emotion in a conversation but also pinpointing the specific utterance that caused that emotion.

### Key Components:
1. **Text Analysis**: Utilizing RoBERTa-base for semantic understanding of utterances.
2. **Audio Analysis**: Extracting features using Wav2Vec 2.0 to capture tonal and emotional cues from speech.
3. **Multimodal Fusion**: Integrating text and audio embeddings for joint classification.
4. **Causal Distance Prediction**: Modeling the distance between an emotion and its cause.

---
## 1. Setup and Initialization
We begin by installing necessary libraries, importing dependencies, and mounting Google Drive to access the dataset.

In [ ]:
%pip install pandas matplotlib seaborn librosa wordcloud

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import os
import numpy as np
from wordcloud import WordCloud


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [ ]:
BASE_PATH = './'

In [ ]:
MELD_CSV_PATH = os.path.join(BASE_PATH, 'train_sent_emo.csv')
ECAC_JSON_PATH = os.path.join(BASE_PATH, 'Subtask_2_train.json')
AUDIO_FOLDER = os.path.join(BASE_PATH, 'train_splits')

## 2. Exploratory Data Analysis (EDA)
Understanding the distribution of emotions and the causal relationship between utterances is crucial for modeling.

In [ ]:
df_text = pd.read_csv(MELD_CSV_PATH)
print(f"Text Data Loaded: {len(df_text)} utterances")

In [ ]:
with open(ECAC_JSON_PATH, 'r') as f:
    data_cause = json.load(f)
print(f"Cause Data Loaded: {len(data_cause)} conversations")

In [ ]:
cause_pairs = []

for conv_data in data_cause:
    if isinstance(conv_data, dict):
        conv_id = conv_data.get('conversation_ID')

        if 'emotion-cause_pairs' in conv_data and isinstance(conv_data['emotion-cause_pairs'], list):
            for pair_str_list in conv_data['emotion-cause_pairs']:
                if len(pair_str_list) == 2:
                    try:

                        emotion_utterance_key = pair_str_list[0]
                        cause_utterance_key = pair_str_list[1]

                        emo_id = int(emotion_utterance_key.split('_')[0].replace('utt', ''))
                        cause_id = int(cause_utterance_key.split('_')[0].replace('utt', ''))

                        emotion_type = 'unknown'
                        if 'conversation' in conv_data and isinstance(conv_data['conversation'], list):
                            for utterance_data in conv_data['conversation']:

                                if utterance_data.get('utterance_ID') == emo_id:
                                    emotion_type = utterance_data.get('emotion', 'unknown')
                                    break

                        pair_data = {
                            'conv_id': conv_id,
                            'emotion_type': emotion_type,
                            'distance': emo_id - cause_id,
                            'emotion_id': emo_id,
                            'cause_id': cause_id
                        }
                        cause_pairs.append(pair_data)
                    except (ValueError, IndexError, KeyError):

                        continue

In [ ]:
df_cause = pd.DataFrame(cause_pairs)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(
    df_cause['distance'],
    bins=np.arange(0, 15) - 0.5,
    kde=False,
    color='cornflowerblue',
    edgecolor='black'
)

plt.title('Causal Lag Analysis: How far back is the Cause?', fontsize=16)
plt.xlabel('Distance (Utterances)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(range(0, 11))
plt.axvline(x=3, color='darkgreen', linestyle='--', linewidth=2,
            label='Proposed Window Size (3)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Avg Distance: {df_cause['distance'].mean():.2f} turns")


### 2.1 Emotion Distribution and Text Analysis
We visualize the frequency of each emotion class and generate word clouds to see the most common words associated with specific emotions.

In [ ]:
plt.figure(figsize=(10, 6))
order = df_text['Emotion'].value_counts().index
sns.countplot(data=df_text, x='Emotion', order=order)
plt.title('Emotion Class Distribution (Imbalance Check)', fontsize=16)
plt.ylabel('Count')
plt.show()

In [ ]:
text_anger = " ".join(df_text[df_text['Emotion'] == 'anger']['Utterance'].astype(str))
wc = WordCloud(width=800, height=400, background_color='black').generate(text_anger)

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud for Class: ANGER')
plt.show()

In [ ]:
text_joy = " ".join(df_text[df_text['Emotion'] == 'joy']['Utterance'].astype(str))
wc = WordCloud(width=800, height=400, background_color='white').generate(text_joy)
plt.figure(figsize=(10, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud for Class: ANGER')
plt.show()

In [ ]:
import pandas as pd
import os

if not os.path.exists(MELD_CSV_PATH):
    print(f"\nError: File not found at {MELD_CSV_PATH}")
    print(f"Please ensure 'train_sent_emo.csv' is present in '{BASE_PATH}'.")

    print(f"Contents of {BASE_PATH}: {os.listdir(BASE_PATH)}")
else:
    df_text = pd.read_csv(MELD_CSV_PATH)
    print(f"Text Data Loaded: {len(df_text)} utterances")

    random_utterance = df_text.sample(n=1, random_state=42)
    print("\nSelected Random Utterance:")
    print(random_utterance)


In [ ]:
dialogue_id = random_utterance['Dialogue_ID'].iloc[0]
utterance_id = random_utterance['Utterance_ID'].iloc[0]

AUDIO_FOLDER = os.path.join(BASE_PATH, 'train_splits')

audio_filename = f"dia{dialogue_id}_utt{utterance_id}.mp4"
full_audio_path = os.path.join(AUDIO_FOLDER, audio_filename)

print(f"Dialogue ID: {dialogue_id}")
print(f"Utterance ID: {utterance_id}")
print(f"Audio Folder: {AUDIO_FOLDER}")
print(f"Constructed Audio Filename: {audio_filename}")
print(f"Full Audio File Path: {full_audio_path}")

In [ ]:
import librosa

y, sr = librosa.load(full_audio_path, sr=None)

utterance_text = random_utterance['Utterance'].iloc[0]

print(f"Audio loaded: Time series shape {y.shape}, Sampling Rate {sr} Hz")
print(f"Extracted Utterance Text: {utterance_text}")

In [ ]:
import numpy as np

audio_duration = len(y) / sr

char_count = len(utterance_text)

word_count = len(utterance_text.split())

print(f"\n--- Analysis of Sampled Utterance ---")
print(f"Audio Duration: {audio_duration:.2f} seconds")
print(f"Utterance Text Character Count: {char_count}")
print(f"Utterance Text Word Count: {word_count}")

In [ ]:
audio_durations = []
missing_files = 0

check_limit = 200
counter = 0

print(f"Checking Audio/Video files in {AUDIO_FOLDER}...")

In [ ]:

# !apt-get install -y ffmpeg

# !pip install librosa soundfile --upgrade

In [ ]:
# for index, row in df_text.iterrows():
#     if counter >= check_limit: break

#     filename = f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.mp4"
#     filepath = os.path.join(AUDIO_FOLDER, filename)

#     if os.path.exists(filepath):
#         try:

#             dur = librosa.get_duration(path=filepath)
#             audio_durations.append({'Emotion': row['Emotion'], 'Duration': dur})
#             counter += 1
#         except Exception as e:

#             pass
#     else:
#         missing_files += 1

In [ ]:
# if len(audio_durations) > 0:
#     df_audio = pd.DataFrame(audio_durations)

#     plt.figure(figsize=(10, 6))
#     sns.boxplot(x='Emotion', y='Duration', data=df_audio)
#     plt.title('Audio Clip Duration Distribution by Emotion')
#     plt.ylabel('Duration (Seconds)')
#     plt.show()

#     print(f"Analyzed {len(df_audio)} audio files.")
# else:
#     print("Skipped Audio Plot: No audio files found in the specified path.")

# print(f"EDA Complete.")

In [ ]:

# import numpy as np
# from tqdm import tqdm

# sampled_df = df_text.groupby('Emotion').apply(lambda x: x.sample(n=50, random_state=42)).reset_index(drop=True)

# audio_energies = []

# print(f"Analyzing Audio Energy for {len(sampled_df)} samples...")

# for index, row in tqdm(sampled_df.iterrows(), total=sampled_df.shape[0]):

#     filename = f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.mp4"
#     filepath = os.path.join(AUDIO_FOLDER, filename)

#     if os.path.exists(filepath):
#         try:

#             y, sr = librosa.load(filepath, sr=16000, duration=3.0)

#             rms = librosa.feature.rms(y=y)
#             avg_energy = np.mean(rms)

#             audio_energies.append({
#                 'Emotion': row['Emotion'],
#                 'Energy_RMS': avg_energy
#             })
#         except:
#             continue

# if len(audio_energies) > 0:
#     df_energy = pd.DataFrame(audio_energies)

#     plt.figure(figsize=(10, 6))

#     sns.boxplot(x='Emotion', y='Energy_RMS', data=df_energy, palette='magma', order=order)
#     plt.title('Cross-Modal Correlation: Audio Energy (Loudness) vs. Emotion')
#     plt.ylabel('Average RMS Energy')
#     plt.show()


#     print("\nMean Energy by Emotion (Higher = Louder):")
#     print(df_energy.groupby('Emotion')['Energy_RMS'].mean().sort_values(ascending=False))
# else:
#     print("Could not analyze energy (No audio files found).")

## 3. Audio Feature Extraction
In this section, we process the multimodal aspect of the dataset. We use **Wav2Vec 2.0** to extract acoustic features from the conversation audio files.

In [ ]:
# !pip install moviepy transformers librosa torch tqdm


In [ ]:
# import os
# import torch
# import librosa
# import numpy as np
# import pickle
# from moviepy.editor import VideoFileClip
# from transformers import Wav2Vec2Processor, Wav2Vec2Model
# from tqdm import tqdm


# BASE_PATH = '.'


# VIDEO_FOLDER = os.path.join(BASE_PATH, 'train_splits')


# WAV_FOLDER = os.path.join(BASE_PATH, 'temp_wavs')
# os.makedirs(WAV_FOLDER, exist_ok=True)


# FEATURE_SAVE_PATH = os.path.join(BASE_PATH, 'audio_features.pkl')


# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")


# print("Loading Wav2Vec 2.0 Model...")
# processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
# model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device)
# model.eval()


# audio_features_dict = {}
# error_files = []


# video_files = [f for f in os.listdir(VIDEO_FOLDER) if f.endswith('.mp4')]
# print(f"Found {len(video_files)} video files. Starting extraction...")

# for video_file in tqdm(video_files):
#     video_path = os.path.join(VIDEO_FOLDER, video_file)
#     wav_path = os.path.join(WAV_FOLDER, video_file.replace('.mp4', '.wav'))
#     file_id = video_file.replace('.mp4', '')

#     try:
#         if not os.path.exists(wav_path):
#             video = VideoFileClip(video_path)

#             video.audio.write_audiofile(wav_path, fps=16000, nbytes=2, codec='pcm_s16le', verbose=False, logger=None)
#             video.close()


#         audio_input, sr = librosa.load(wav_path, sr=16000, duration=6.0)


#         input_values = processor(audio_input, sampling_rate=16000, return_tensors="pt", padding="longest").input_values
#         input_values = input_values.to(device)

#         with torch.no_grad():
#             outputs = model(input_values)

#         last_hidden_state = outputs.last_hidden_state
#         pooled_output = torch.mean(last_hidden_state, dim=1).squeeze().cpu().numpy()

#         audio_features_dict[file_id] = pooled_output

#         if os.path.exists(wav_path):
#             os.remove(wav_path)

#     except Exception as e:
#         print(f"Error processing {video_file}: {e}")
#         error_files.append(video_file)

# print(f"\nExtraction Complete! Processed {len(audio_features_dict)} files.")
# print(f"Errors: {len(error_files)}")

# print(f"Saving features to {FEATURE_SAVE_PATH}...")
# with open(FEATURE_SAVE_PATH, 'wb') as f:
#     pickle.dump(audio_features_dict, f)

In [ ]:
# import os
# import torch
# import librosa
# import numpy as np
# import pickle
# from moviepy.editor import VideoFileClip
# from transformers import Wav2Vec2Processor, Wav2Vec2Model
# from tqdm import tqdm


# BASE_PATH = '.'

# VIDEO_FOLDER = os.path.join(BASE_PATH, 'dev_splits_complete')

# WAV_FOLDER = os.path.join(BASE_PATH, 'temp_wavs_dev')
# os.makedirs(WAV_FOLDER, exist_ok=True)

# FEATURE_SAVE_PATH = os.path.join(BASE_PATH, 'dev_audio_features.pkl')

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# print("Loading Wav2Vec 2.0 Model...")
# processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
# model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device)
# model.eval()

# audio_features_dict = {}
# error_files = []

# video_files = [f for f in os.listdir(VIDEO_FOLDER) if f.endswith('.mp4')]
# print(f"Found {len(video_files)} video files. Starting extraction...")

# for video_file in tqdm(video_files):
#     video_path = os.path.join(VIDEO_FOLDER, video_file)
#     wav_path = os.path.join(WAV_FOLDER, video_file.replace('.mp4', '.wav'))
#     file_id = video_file.replace('.mp4', '')

#     try:

#         if not os.path.exists(wav_path):
#             video = VideoFileClip(video_path)
#             video.audio.write_audiofile(wav_path, fps=16000, nbytes=2, codec='pcm_s16le', verbose=False, logger=None)
#             video.close()

#         audio_input, sr = librosa.load(wav_path, sr=16000, duration=6.0)

#         input_values = processor(audio_input, sampling_rate=16000, return_tensors="pt", padding="longest").input_values
#         input_values = input_values.to(device)

#         with torch.no_grad():
#             outputs = model(input_values)

#         last_hidden_state = outputs.last_hidden_state
#         pooled_output = torch.mean(last_hidden_state, dim=1).squeeze().cpu().numpy()

#         audio_features_dict[file_id] = pooled_output

#         if os.path.exists(wav_path):
#             os.remove(wav_path)

#     except Exception as e:
#         error_files.append(video_file)

# print(f"\nExtraction Complete! Processed {len(audio_features_dict)} files.")
# print(f"Errors: {len(error_files)}")

# print(f"Saving features to {FEATURE_SAVE_PATH}...")
# with open(FEATURE_SAVE_PATH, 'wb') as f:
#     pickle.dump(audio_features_dict, f)

In [ ]:
import json
import pandas as pd
import os

# CONFIG
BASE_PATH = '.'
JSON_PATH = os.path.join(BASE_PATH, 'Subtask_2_train.json') # Your file
CSV_PATH = os.path.join(BASE_PATH, 'train_sent_emo.csv')

# 1. CHECK RAW JSON COUNTS
print(f"--- INSPECTING JSON: {os.path.basename(JSON_PATH)} ---")
with open(JSON_PATH, 'r') as f:
    data = json.load(f)

total_conv = len(data)
total_pairs = 0
for item in data:
    if 'emotion-cause_pairs' in item:
        total_pairs += len(item['emotion-cause_pairs'])

print(f"Total Conversations: {total_conv}")
print(f"Total Raw Pairs in JSON: {total_pairs}")
print("------------------------------------------------")

# 2. CHECK MAPPING LOGIC
print("--- TESTING MAPPING LOGIC ---")
df = pd.read_csv(CSV_PATH)
print(f"CSV Entries: {len(df)}")

mapped_count = 0
failed_count = 0

# Sample failure to debug
sample_fail = ""

for item in data:
    if 'emotion-cause_pairs' in item and 'conversation' in item:
        # Build local map from video_name
        id_to_key = {}
        for utt in item['conversation']:
            u_id = utt['utterance_ID']
            vid = utt['video_name'] # e.g. "dia1utt1.mp4"

            # PARSING LOGIC
            try:
                clean = vid.replace('.mp4', '')
                parts = clean.split('utt')
                d_num = int(parts[0].replace('dia', '')) - 1 # Convert to 0-based
                u_num = int(parts[1]) - 1

                real_key = f"dia{d_num}_utt{u_num}"
                id_to_key[u_id] = real_key
            except:
                pass

        # Check Pairs
        for pair in item['emotion-cause_pairs']:
            try:
                # pair = ["3_joy", "1"]
                e_raw = int(pair[0].split('_')[0])
                c_raw = int(pair[1])

                if e_raw in id_to_key and c_raw in id_to_key:
                    mapped_count += 1
                else:
                    failed_count += 1
                    if failed_count == 1:
                        sample_fail = f"Failed to map JSON IDs {e_raw},{c_raw} using Video Names: {list(id_to_key.values())[:3]}"
            except:
                failed_count += 1

print(f"Successfully Mapped: {mapped_count}")
print(f"Failed to Map:       {failed_count}")

if mapped_count > 5000:
    print("\n✅ DIAGNOSIS: The logic works! You can use this file.")
else:
    print("\n❌ DIAGNOSIS: Logic Failure.")
    print(f"Debug Info: {sample_fail}")

## 4. Custom Dataset and DataLoader
The `MECPEDataset` class handles the integration of:
- **Tokenizer labels** from RoBERTa.
- **Pre-computed audio vectors**.
- **Causal lag mapping** from the JSON annotations.

## 5. Model Architecture: Dual-Stream Multimodal MECPE

This model implements a cross-modal attention mechanism to fuse text and audio features:
- **Text Stream**: RoBERTa-base with the first 6 layers frozen for efficiency.
- **Audio Stream**: A fully connected network processing pre-computed Wav2Vec features.
- **Cross-Attention**: Allows the text representation to attend to audio cues.
- **Bi-LSTM**: Captures contextual dependencies from the fused features.
- **Multi-Task Heads**: Decodes both emotion classes and causal lag distances.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel, get_linear_schedule_with_warmup
import pandas as pd
import pickle
import json
import os
import numpy as np
from tqdm import tqdm
from torch.optim import AdamW
from collections import Counter
from sklearn.metrics import f1_score, accuracy_score

# ================= 1. CONFIGURATION =================
CONFIG = {
    'epochs': 15,
    'lr': 3e-5,              # Base LR for BERT
    'head_lr': 3e-4,         # Higher LR for classification heads
    'batch_size': 16,
    'max_len': 160,          # Large enough for 3 sentences
    'weight_decay': 0.1,
    'base_path': '.',
    'model_save_path': 'best_model.pth'
}

EMOTION_MAP = {
    'neutral': 0, 'joy': 1, 'surprise': 2, 'anger': 3,
    'sadness': 4, 'disgust': 5, 'fear': 6
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

# ================= 2. DATASET (FIXED ID MAPPING) =================
class MECPEDataset(Dataset):
    def __init__(self, csv_path, json_path, audio_pkl_path, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.df = pd.read_csv(csv_path)

        with open(audio_pkl_path, 'rb') as f:
            self.audio_data = pickle.load(f)

        with open(json_path, 'r') as f:
            self.cause_data = json.load(f)

        self.cause_map = {}
        mapped_count = 0
        
        # --- ROBUST TEXT-BASED ALIGNMENT ---
        # 1. Group CSV conversations by Dialogue_ID
        csv_convs = {}
        for idx, row in self.df.iterrows():
            did = row['Dialogue_ID']
            if did not in csv_convs: csv_convs[did] = []
            text = str(row['Utterance']).strip().lower()
            text = "".join([c for c in text if c.isalnum()]) # Filter punctuation
            csv_convs[did].append( (row['Utterance_ID'], text) )
            
        # Map first utterance text to list of Match Candidates (Dialogue IDs)
        first_utt_map = {}
        for did, utts in csv_convs.items():
            if utts:
                utts.sort(key=lambda x: x[0])
                first_text = utts[0][1]
                k = first_text[:50]
                if k not in first_utt_map: first_utt_map[k] = []
                first_utt_map[k].append(did)

        # 2. Iterate JSON and find matches in CSV
        for item in self.cause_data:
            json_utts = item['conversation']
            if not json_utts: continue
            
            # Normalize first text
            j_text0 = str(json_utts[0]['text']).strip().lower()
            j_text0 = "".join([c for c in j_text0 if c.isalnum()])
            
            candidates = first_utt_map.get(j_text0[:50], [])
            
            matched_did = None
            for cand_did in candidates:
                csv_u = csv_convs[cand_did]
                match_count = 0
                check_len = min(len(json_utts), len(csv_u))
                if check_len == 0: continue
                
                for i in range(check_len):
                    ct = csv_u[i][1]
                    jt = str(json_utts[i]['text']).strip().lower()
                    jt = "".join([c for c in jt if c.isalnum()])
                    if ct == jt: match_count += 1
                
                if match_count / check_len > 0.5: # Confirm match
                    matched_did = cand_did
                    break
            
            if matched_did is not None:
                # 3. Map JSON Utterance IDs to CSV Utterance IDs
                csv_u_list = csv_convs[matched_did]
                j_id_to_c_id = {}
                for i in range(min(len(json_utts), len(csv_u_list))):
                    j_id_to_c_id[json_utts[i]['utterance_ID']] = csv_u_list[i][0]
                
                if 'emotion-cause_pairs' in item:
                    for pair in item['emotion-cause_pairs']:
                        try:
                            e_raw, c_raw = int(pair[0].split('_')[0]), int(pair[1])
                            e_csv_id = j_id_to_c_id.get(e_raw)
                            
                            # Calculate Dist
                            e_idx, c_idx = -1, -1
                            for idx, u in enumerate(json_utts):
                                if u['utterance_ID'] == e_raw: e_idx = idx
                                if u['utterance_ID'] == c_raw: c_idx = idx
                            
                            if e_idx != -1 and c_idx != -1 and 0 <= (e_idx - c_idx) <= 5 and e_csv_id is not None:
                                key = f"dia{matched_did}_utt{e_csv_id}"
                                self.cause_map[key] = e_idx - c_idx
                                mapped_count += 1
                        except: pass
        
        print(f"✅ Dataset Loaded: Mapped {mapped_count} cause labels via matched IDs.")

    def __len__(self): return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        dia_id, utt_id = row['Dialogue_ID'], row['Utterance_ID']
        # Context logic...
        context = []
        for i in [2, 1]:
            prev_idx = index - i
            if prev_idx >= 0:
                prev_row = self.df.iloc[prev_idx]
                if prev_row['Dialogue_ID'] == dia_id:
                    context.append(str(prev_row['Utterance']))

        full_text = f" {self.tokenizer.sep_token} ".join(context + [str(row['Utterance'])])
        inputs = self.tokenizer(full_text, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')

        unique_id = f"dia{dia_id}_utt{utt_id}"
        audio_vec = torch.tensor(self.audio_data.get(unique_id, np.zeros(768)), dtype=torch.float32)

        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'audio_vec': audio_vec,
            'emotion_label': torch.tensor(EMOTION_MAP.get(row['Emotion'].lower(), 0), dtype=torch.long),
            'cause_label': torch.tensor(self.cause_map.get(unique_id, -1), dtype=torch.long)
        }

# ================= 3. MODEL (SEQUENCE-AWARE) =================
class DualStreamMECPE(nn.Module):
    def __init__(self, num_emotions=7, window_size=6):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        # Unfreeze half of RoBERTa
        for layer in list(self.roberta.encoder.layer)[:6]:
            for param in layer.parameters(): param.requires_grad = False

        self.audio_fc = nn.Sequential(nn.Linear(768, 768), nn.ReLU(), nn.Dropout(0.3))
        self.cross_attention = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)

        # Bi-LSTM over the sequence
        self.lstm = nn.LSTM(input_size=768, hidden_size=256, num_layers=1, batch_first=True, bidirectional=True)

        self.emotion_head = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, num_emotions))
        self.cause_head = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, window_size))

    def forward(self, input_ids, attention_mask, audio_vec):
        text_seq = self.roberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        audio_emb = self.audio_fc(audio_vec).unsqueeze(1)

        # Cross-Attention Fusion
        fused, _ = self.cross_attention(query=text_seq, key=audio_emb, value=audio_emb)
        fused = text_seq + fused # Residual connection

        # Temporal/Contextual Processing
        lstm_out, _ = self.lstm(fused)
        # Max-pool over sequence length
        feat, _ = torch.max(lstm_out, dim=1)

        return self.emotion_head(feat), self.cause_head(feat)

# ================= 4. TRAINING =================
def run_train():
    # Initialize
    train_ds = MECPEDataset(os.path.join(CONFIG['base_path'], 'train_sent_emo.csv'),
                            os.path.join(CONFIG['base_path'], 'Subtask_2_train.json'),
                            os.path.join(CONFIG['base_path'], 'audio_features.pkl'), tokenizer, CONFIG['max_len'])
    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)

    # Validation Loader (uses dev.json)
    dev_ds = MECPEDataset(os.path.join(CONFIG['base_path'], 'dev_sent_emo.csv'),
                          os.path.join(CONFIG['base_path'], 'dev.json'),
                          os.path.join(CONFIG['base_path'], 'dev_audio_features.pkl'), tokenizer, CONFIG['max_len'])
    dev_loader = DataLoader(dev_ds, batch_size=32, shuffle=False)

    model = DualStreamMECPE().to(device)

    # Differential Learning Rates
    bert_params = [p for n, p in model.named_parameters() if "roberta" in n]
    head_params = [p for n, p in model.named_parameters() if "roberta" not in n]
    optimizer = AdamW([{'params': bert_params, 'lr': CONFIG['lr']},
                       {'params': head_params, 'lr': CONFIG['head_lr']}], weight_decay=CONFIG['weight_decay'])

    # Smoothed Weights
    all_c = [train_ds[i]['cause_label'].item() for i in range(len(train_ds))]
    counts = Counter([c for c in all_c if c != -1])
    weights = torch.tensor([np.sqrt(sum(counts.values())/counts.get(i, 1)) for i in range(6)]).float().to(device)

    criterion_e = nn.CrossEntropyLoss()
    criterion_c = nn.CrossEntropyLoss(weight=weights, ignore_index=-1)

    best_score = 0
    for epoch in range(CONFIG['epochs']):
        model.train()
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            ids, mask, audio = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['audio_vec'].to(device).float()
            lbl_e, lbl_c = batch['emotion_label'].to(device), batch['cause_label'].to(device)

            optimizer.zero_grad()
            out_e, out_c = model(ids, mask, audio)
            loss = (0.3 * criterion_e(out_e, lbl_e)) + (0.7 * criterion_c(out_c, lbl_c))
            loss.backward()
            optimizer.step()

        # Validation logic
        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for batch in dev_loader:
                lbl_c = batch['cause_label'].to(device)
                _, out_c = model(batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['audio_vec'].to(device).float())
                valid = lbl_c != -1
                if valid.sum() > 0:
                    preds.extend(torch.argmax(out_c, dim=1)[valid].cpu().numpy())
                    labels.extend(lbl_c[valid].cpu().numpy())

        f1 = f1_score(labels, preds, average='macro')
        print(f"📊 Epoch {epoch+1} Cause F1: {f1:.4f} | Predictions: {dict(Counter(preds))}")

        if f1 > best_score:
            best_score = f1
            torch.save(model.state_dict(), os.path.join(CONFIG['base_path'], CONFIG['model_save_path']))

run_train()

## 6. Training and Optimization

To handle class imbalance in causal lags, we calculate **class weights** based on their frequency in the training data. The training loop optimizes a weighted sum of Emotion Cross-Entropy and Cause Cross-Entropy.

## 7. Performance Evaluation

We evaluate the model's performance using Classification Reports and Confusion Matrices for both tasks.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# 1. LOAD THE BEST SAVED MODEL
model_path = os.path.join(CONFIG['base_path'], CONFIG['model_save_path'])
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path))
    print("✅ Loaded Best Model for Evaluation.")

# 2. SET EVALUATION TARGET (Switch to dev_loader for real results!)
TARGET_LOADER = dev_loader if dev_loader is not None else train_loader
TITLE_SUFFIX = "(Validation Set)" if dev_loader is not None else "(Training Set)"

model.eval()
all_preds_emo, all_labels_emo = [], []
all_preds_cause, all_labels_cause = [], []

print(f"Generating Metrics for {TITLE_SUFFIX}...")

with torch.no_grad():
    for batch in tqdm(TARGET_LOADER):
        ids, mask, audio = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['audio_vec'].to(device).float()
        lbl_e, lbl_c = batch['emotion_label'].to(device), batch['cause_label'].to(device)

        out_e, out_c = model(ids, mask, audio)

        p_e = torch.argmax(out_e, dim=1)
        p_c = torch.argmax(out_c, dim=1)

        all_preds_emo.extend(p_e.cpu().numpy())
        all_labels_emo.extend(lbl_e.cpu().numpy())

        valid_mask = lbl_c != -1
        if valid_mask.sum() > 0:
            all_preds_cause.extend(p_c[valid_mask].cpu().numpy())
            all_labels_cause.extend(lbl_c[valid_mask].cpu().numpy())

# --- EMOTION REPORT ---
print("\n" + "="*40)
print(f"1. EMOTION RECOGNITION {TITLE_SUFFIX}")
print("="*40)

# Ensure EMOTION_MAP is defined (assuming 7 emotions based on standard datasets)
# If you don't have it, use: emotion_names = [str(i) for i in range(7)]
emotion_names = list(EMOTION_MAP.keys()) if 'EMOTION_MAP' in locals() else [str(i) for i in range(7)]
print(classification_report(all_labels_emo, all_preds_emo, target_names=emotion_names))

# --- EMOTION CONFUSION MATRIX ---
plt.figure(figsize=(10, 8))
cm_emo = confusion_matrix(all_labels_emo, all_preds_emo)
cm_norm_emo = cm_emo.astype('float') / cm_emo.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm_emo, annot=True, fmt='.2f', cmap='Blues', xticklabels=emotion_names, yticklabels=emotion_names)
plt.title(f'Emotion Confusion Matrix {TITLE_SUFFIX}')
plt.show()

# --- CAUSE REPORT ---
print("\n" + "="*40)
print(f"2. CAUSAL SPAN EXTRACTION {TITLE_SUFFIX}")
print("="*40)

if len(all_labels_cause) > 0:
    unique_labels = sorted(list(set(all_labels_cause)))
    # Labels are 0 to 5 (Lags)
    target_names = [f"Lag {i}" for i in unique_labels]

    print(classification_report(all_labels_cause, all_preds_cause, labels=unique_labels, target_names=target_names))

    # --- CAUSE CONFUSION MATRIX ---
    plt.figure(figsize=(10, 8))
    cm_cause = confusion_matrix(all_labels_cause, all_preds_cause)
    cm_norm_cause = cm_cause.astype('float') / cm_cause.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm_cause, annot=True, fmt='.2f', cmap='Greens', xticklabels=target_names, yticklabels=target_names)
    plt.title(f'Causal Lag Confusion Matrix {TITLE_SUFFIX}')
    plt.show()

In [ ]:
# from sklearn.metrics import confusion_matrix, classification_report
# import seaborn as sns
# import matplotlib.pyplot as plt
# import numpy as np
# from tqdm import tqdm

# TARGET_LOADER = train_loader
# TITLE_SUFFIX = "(Training Set)"

# model.eval()
# all_preds_emo = []
# all_labels_emo = []
# all_preds_cause = []
# all_labels_cause = []

# print(f"Generating Full Metrics for {TITLE_SUFFIX}...")

# with torch.no_grad():
#     for batch in tqdm(TARGET_LOADER):
#         input_ids = batch['input_ids'].to(device)
#         mask = batch['attention_mask'].to(device)
#         audio_vec = batch['audio_vec'].to(device).float()

#         label_emo = batch['emotion_label'].to(device)
#         label_cause = batch['cause_label'].to(device)

#         out_emo, out_cause = model(input_ids, mask, audio_vec)

#         _, pred_e = torch.max(out_emo, 1)
#         _, pred_c = torch.max(out_cause, 1)

#         all_preds_emo.extend(pred_e.cpu().numpy())
#         all_labels_emo.extend(label_emo.cpu().numpy())

#         valid_mask = label_cause != -1
#         if valid_mask.sum() > 0:
#             all_preds_cause.extend(pred_c[valid_mask].cpu().numpy())
#             all_labels_cause.extend(label_cause[valid_mask].cpu().numpy())

# print("\n" + "="*40)
# print(f"1. EMOTION RECOGNITION {TITLE_SUFFIX}")
# print("="*40)

# emotion_names = list(EMOTION_MAP.keys())
# print(classification_report(all_labels_emo, all_preds_emo, target_names=emotion_names))

# plt.figure(figsize=(8, 6))
# cm_emo = confusion_matrix(all_labels_emo, all_preds_emo)
# with np.errstate(divide='ignore', invalid='ignore'):
#     cm_norm_emo = cm_emo.astype('float') / cm_emo.sum(axis=1)[:, np.newaxis]
# cm_norm_emo = np.nan_to_num(cm_norm_emo)

# sns.heatmap(cm_norm_emo, annot=True, fmt='.2f', cmap='Blues',
#             xticklabels=emotion_names, yticklabels=emotion_names)
# plt.title(f'Emotion Confusion Matrix {TITLE_SUFFIX}')
# plt.ylabel('True Label')
# plt.xlabel('Predicted Label')
# plt.show()

In [ ]:
# print("\n" + "="*40)
# print(f"2. CAUSAL SPAN EXTRACTION {TITLE_SUFFIX}")
# print("="*40)

# if len(all_labels_cause) > 0:
#     unique_labels = sorted(list(set(all_labels_cause)))
#     target_names = [f"Lag {i}" for i in unique_labels]

#     print(classification_report(all_labels_cause, all_preds_cause,
#                                 labels=unique_labels, target_names=target_names))

#     plt.figure(figsize=(10, 8))
#     cm_cause = confusion_matrix(all_labels_cause, all_preds_cause)

#     with np.errstate(divide='ignore', invalid='ignore'):
#         cm_norm_cause = cm_cause.astype('float') / cm_cause.sum(axis=1)[:, np.newaxis]
#     cm_norm_cause = np.nan_to_num(cm_norm_cause)

#     sns.heatmap(cm_norm_cause, annot=True, fmt='.2f', cmap='Greens',
#                 xticklabels=target_names, yticklabels=target_names)
#     plt.title(f'Causal Lag Confusion Matrix {TITLE_SUFFIX}')
#     plt.ylabel('True Label')
#     plt.xlabel('Predicted Label')
#     plt.show()
# else:
#     print("❌ No valid cause labels found.")

## 8. Model Summary
A structural overview of the parameters and layers in the final model.

In [30]:
print(model)

NameError: name 'model' is not defined

In [ ]:
 %pip install torchinfo

In [31]:
from torchinfo import summary
import torch

model = DualStreamMECPE(num_emotions=7, window_size=6)

dummy_ids = torch.zeros((1, 64), dtype=torch.long)
dummy_mask = torch.zeros((1, 64), dtype=torch.long)

dummy_audio = torch.zeros((1, 768), dtype=torch.float)

print("Generating Summary for DualStreamMECPE...")
summary(model, input_data=[dummy_ids, dummy_mask, dummy_audio])

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Generating Summary for DualStreamMECPE...


Layer (type:depth-idx)                                            Output Shape              Param #
DualStreamMECPE                                                   [1, 7]                    --
├─RobertaModel: 1-1                                               [1, 768]                  --
│    └─RobertaEmbeddings: 2-1                                     [1, 64, 768]              --
│    │    └─Embedding: 3-1                                        [1, 64, 768]              38,603,520
│    │    └─Embedding: 3-2                                        [1, 64, 768]              768
│    │    └─Embedding: 3-3                                        [1, 64, 768]              394,752
│    │    └─LayerNorm: 3-4                                        [1, 64, 768]              1,536
│    │    └─Dropout: 3-5                                          [1, 64, 768]              --
│    └─RobertaEncoder: 2-2                                        [1, 64, 768]              --
│    │    └─ModuleList: 3-6 

## 9. Preparing for External Testing

Extracting features from the test set to evaluate generalization performance.

In [ ]:
import os
import torch
import librosa
import numpy as np
import pickle
from moviepy.editor import VideoFileClip
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from tqdm import tqdm


BASE_PATH = '.'

VIDEO_FOLDER = os.path.join(BASE_PATH, 'test_splits')

WAV_FOLDER = os.path.join(BASE_PATH, 'temp_wavs_test')
os.makedirs(WAV_FOLDER, exist_ok=True)

FEATURE_SAVE_PATH = os.path.join(BASE_PATH, 'audio_test.pkl')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Loading Wav2Vec 2.0 Model...")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device)
model.eval()

audio_features_dict = {}
error_files = []

video_files = [f for f in os.listdir(VIDEO_FOLDER) if f.endswith('.mp4')]
print(f"Found {len(video_files)} video files. Starting extraction...")

for video_file in tqdm(video_files):
    video_path = os.path.join(VIDEO_FOLDER, video_file)
    wav_path = os.path.join(WAV_FOLDER, video_file.replace('.mp4', '.wav'))
    file_id = video_file.replace('.mp4', '')

    try:

        if not os.path.exists(wav_path):
            video = VideoFileClip(video_path)
            video.audio.write_audiofile(wav_path, fps=16000, nbytes=2, codec='pcm_s16le', verbose=False, logger=None)
            video.close()

        audio_input, sr = librosa.load(wav_path, sr=16000, duration=6.0)

        input_values = processor(audio_input, sampling_rate=16000, return_tensors="pt", padding="longest").input_values
        input_values = input_values.to(device)

        with torch.no_grad():
            outputs = model(input_values)

        last_hidden_state = outputs.last_hidden_state
        pooled_output = torch.mean(last_hidden_state, dim=1).squeeze().cpu().numpy()

        audio_features_dict[file_id] = pooled_output

        if os.path.exists(wav_path):
            os.remove(wav_path)

    except Exception as e:
        error_files.append(video_file)

print(f"\nExtraction Complete! Processed {len(audio_features_dict)} files.")
print(f"Errors: {len(error_files)}")

print(f"Saving features to {FEATURE_SAVE_PATH}...")
with open(FEATURE_SAVE_PATH, 'wb') as f:
    pickle.dump(audio_features_dict, f)

### Model Evaluation on Test Set


## 10. Final Evaluation Results

This section executes inference on the unseen test set and provides the final benchmark for:
- **Emotion Recognition Accuracy**: F1-scores and support per emotion.
- **Causal Span Extraction**: Performance across different causal lags.

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from tqdm.auto import tqdm

# ================= CONFIGURATION =================
class Config:
    BASE_PATH = '.'
    TEST_CSV = os.path.join(BASE_PATH, 'test_sent_emo.csv')
    TEST_JSON = os.path.join(BASE_PATH, 'Subtask_2_test.json')
    TEST_AUDIO = os.path.join(BASE_PATH, 'audio_test.pkl')
    MODEL_PATH = os.path.join(BASE_PATH, 'best_model.pth')
    BATCH_SIZE = 32
    MAX_LEN = 64
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def check_paths():
    paths = [Config.TEST_CSV, Config.TEST_JSON, Config.TEST_AUDIO, Config.MODEL_PATH]
    for p in paths:
        if not os.path.exists(p):
            raise FileNotFoundError(f"❌ Missing: {p}")
    print(f"✅ Test files found. Running on {Config.DEVICE}...")

def load_model(path, device):
    """Loads weights into the pre-defined model instance."""
    print("🔄 Loading best model weights...")
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    model.eval()
    return model

def plot_cm(y_true, y_pred, title, labels, cmap='Blues'):
    """Plots a normalized confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    with np.errstate(divide='ignore', invalid='ignore'):
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    cm_norm = np.nan_to_num(cm_norm)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap=cmap,
                xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(title)
    plt.show()

def run_test_inference():
    check_paths()

    # Initialize Test Dataset
    test_ds = MECPEDataset(Config.TEST_CSV, Config.TEST_JSON, Config.TEST_AUDIO, tokenizer, Config.MAX_LEN)
    test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False)

    # Load Model
    eval_model = load_model(Config.MODEL_PATH, Config.DEVICE)

    results = {
        "emo_true": [], "emo_pred": [],
        "cause_true": [], "cause_pred": []
    }

    print("🔎 Running Inference on Test Set...")
    with torch.no_grad():
        for batch in tqdm(test_loader):
            ids = batch['input_ids'].to(Config.DEVICE)
            mask = batch['attention_mask'].to(Config.DEVICE)
            audio = batch['audio_vec'].to(Config.DEVICE).float()
            lbl_e = batch['emotion_label'].to(Config.DEVICE)
            lbl_c = batch['cause_label'].to(Config.DEVICE)

            out_e, out_c = eval_model(ids, mask, audio)

            p_e = torch.argmax(out_e, dim=1)
            p_c = torch.argmax(out_c, dim=1)

            results["emo_true"].extend(lbl_e.cpu().numpy())
            results["emo_pred"].extend(p_e.cpu().numpy())

            valid_mask = lbl_c != -1
            if valid_mask.sum() > 0:
                results["cause_true"].extend(lbl_c[valid_mask].cpu().numpy())
                results["cause_pred"].extend(p_c[valid_mask].cpu().numpy())

    return results

# ================= EXECUTION =================
test_results = run_test_inference()

print("\n" + "🏆" + "="*50 + "🏆")
print("             FINAL TEST SET RESULTS")
print("="*52)

# --- EMOTION STATS ---
emo_acc = accuracy_score(test_results['emo_true'], test_results['emo_pred'])
emo_f1 = f1_score(test_results['emo_true'], test_results['emo_pred'], average='macro')

print(f"\n🎬 EMOTION RECOGNITION")
print(f"   - Accuracy: {emo_acc:.2%}")
print(f"   - Macro F1: {emo_f1:.4f} (Primary Metric)")

emo_names = list(EMOTION_MAP.keys())
print(classification_report(test_results['emo_true'], test_results['emo_pred'], target_names=emo_names))
plot_cm(test_results['emo_true'], test_results['emo_pred'], "Emotion Confusion Matrix (Test)", emo_names, cmap='Blues')

# --- CAUSE STATS ---
if len(test_results['cause_true']) > 0:
    cause_acc = accuracy_score(test_results['cause_true'], test_results['cause_pred'])
    cause_f1 = f1_score(test_results['cause_true'], test_results['cause_pred'], average='macro')

    print(f"\n🔎 CAUSAL SPAN EXTRACTION")
    print(f"   - Accuracy: {cause_acc:.2%}")
    print(f"   - Macro F1: {cause_f1:.4f} (Primary Metric)")

    unique_labels = sorted(list(set(test_results['cause_true'])))
    cause_names = [f"Lag {i}" for i in unique_labels]

    print(classification_report(test_results['cause_true'], test_results['cause_pred'],
                                labels=unique_labels, target_names=cause_names))

    plot_cm(test_results['cause_true'], test_results['cause_pred'], "Causal Lag Confusion Matrix (Test)", cause_names, cmap='Greens')

    # --- OVERALL SCORE ---
    print(f"\n⭐ OVERALL COMBINED F1: {(emo_f1 + cause_f1)/2:.4f}")
else:
    print("\n❌ No valid cause labels found in Test Set (Blind Set Detected).")

### 10.1 Causal Lag Performance
Detailed metrics for predicted causality distances.

In [ ]:
# %pip install gradio

In [ ]:
%pip install -q git+https://github.com/openai/whisper.git
%pip install -q gradio librosa soundfile transformers

import gradio as gr
import torch
import librosa
import numpy as np
import pandas as pd
import os
import random
import soundfile as sf
import whisper
from transformers import Wav2Vec2Processor, Wav2Vec2Model, RobertaTokenizer

# ================= CONFIGURATION =================
BASE_PATH = '.'
MODEL_PATH = os.path.join(BASE_PATH, 'best_model.pth')
TEST_CSV_PATH = os.path.join(BASE_PATH, 'test_sent_emo.csv')

# Video Folders
POSSIBLE_FOLDERS = [
    os.path.join(BASE_PATH, 'output_repeated_splits_test'),
    os.path.join(BASE_PATH, 'MELD.Raw', 'test_splits'),
    os.path.join(BASE_PATH, 'test_splits')
]
VALID_FOLDERS = [f for f in POSSIBLE_FOLDERS if os.path.exists(f)]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMOTIONS = ['Neutral', 'Joy', 'Surprise', 'Anger', 'Sadness', 'Disgust', 'Fear']

# ================= 1. LOAD ALL AI MODELS =================
print("⏳ Loading Analysis Models (RoBERTa + Wav2Vec)...")
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
audio_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
audio_model_feat = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE)

model = DualStreamMECPE(num_emotions=7, window_size=6)
if torch.cuda.is_available():
    model.load_state_dict(torch.load(MODEL_PATH))
else:
    model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device('cpu')))
model.to(DEVICE)
model.eval()

print("⏳ Loading Whisper (Auto-Transcriber)...")
# Load the 'base' model (good balance of speed/accuracy)
transcriber = whisper.load_model("base")

if os.path.exists(TEST_CSV_PATH):
    df_test = pd.read_csv(TEST_CSV_PATH)
else:
    df_test = pd.DataFrame()

print("✅ All Systems Ready.")

# ================= 2. CORE LOGIC =================
def process_call(audio_path, text_input):
    if audio_path is None:
        return "⚠️ Error: Please speak or upload audio.", {}, ""

    # --- A. AUTO-TRANSCRIPTION (The New Part) ---
    # Only transcribe if text is empty
    if text_input is None or text_input.strip() == "":
        print("🎤 Transcribing Audio with Whisper...")
        try:
            # Whisper expects path or array.
            result = transcriber.transcribe(audio_path)
            text_input = result["text"].strip()
        except Exception as e:
            text_input = "(Transcription Failed)"
            print(f"ASR Error: {e}")

    # --- B. FEATURE EXTRACTION ---
    try:
        # Audio Features
        y, sr = librosa.load(audio_path, sr=16000, duration=6.0)
        inputs = audio_processor(y, sampling_rate=16000, return_tensors="pt", padding="longest")
        input_values = inputs.input_values.to(DEVICE)
        with torch.no_grad():
            outputs = audio_model_feat(input_values)
            audio_vec = torch.mean(outputs.last_hidden_state, dim=1)

        # Text Features
        text_inputs = tokenizer(text_input, max_length=64, padding='max_length', truncation=True, return_tensors='pt')
        input_ids = text_inputs['input_ids'].to(DEVICE)
        mask = text_inputs['attention_mask'].to(DEVICE)

        # --- C. INFERENCE ---
        with torch.no_grad():
            out_emo, out_cause = model(input_ids, mask, audio_vec)

            probs_emo = torch.nn.functional.softmax(out_emo, dim=1)[0]
            pred_emo_idx = torch.argmax(probs_emo).item()
            pred_cause_idx = torch.argmax(out_cause, dim=1).item()

        # --- D. REPORTING ---
        emo_label = EMOTIONS[pred_emo_idx]
        emo_conf = probs_emo[pred_emo_idx].item()

        if pred_cause_idx == 0:
            cause_text = "🔴 SELF-REACTION (Lag 0)\nUser is reacting to the current situation."
        else:
            cause_text = f"⬅️ CONTEXT TRIGGER (Lag {pred_cause_idx})\nTriggered by conversation history ({pred_cause_idx} turns ago)."

        report = f"**Detected Emotion:** {emo_label.upper()} ({emo_conf:.1%})\n\n**Cause:**\n{cause_text}"
        confidences = {EMOTIONS[i]: float(probs_emo[i]) for i in range(len(EMOTIONS))}

        # Return Report, Chart, AND the Transcribed Text (to fill the box)
        return report, confidences, text_input

    except Exception as e:
        return f"Error: {str(e)}", {}, text_input

# ================= 3. SIMULATOR LOGIC =================
def simulate_call():
    if df_test.empty: return None, "No CSV", "Error", {}

    for _ in range(50):
        row = df_test.sample(1).iloc[0]
        fname = f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.mp4"

        found_path = None
        for folder in VALID_FOLDERS:
            path = os.path.join(folder, fname)
            if os.path.exists(path):
                found_path = path
                break

        if found_path:
            # Convert to WAV for browser
            temp_wav = "sim_call.wav"
            y, sr = librosa.load(found_path, sr=16000)
            sf.write(temp_wav, y, sr)

            # Use Ground Truth text for simulation (Faster)
            truth_text = str(row['Utterance'])

            # Run
            report, chart, _ = process_call(temp_wav, truth_text)
            return temp_wav, truth_text, report, chart

    return None, "File not found", "", {}

# ================= 4. UI LAUNCHER =================
with gr.Blocks(title="AI Call Center", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎧 AI Call Center Dashboard (with Auto-Transcription)")

    with gr.Tabs():
        # TAB 1: LIVE
        with gr.TabItem("Live Analysis"):
            gr.Markdown("Speak into the microphone. Leave 'Transcript' empty to use Auto-Detection.")
            with gr.Row():
                live_audio = gr.Audio(type="filepath", sources=["microphone", "upload"])
                live_text = gr.Textbox(label="Transcript (Auto-Fills)")

            btn_live = gr.Button("Analyze Voice", variant="primary")

            with gr.Row():
                out_rep = gr.Markdown()
                out_chart = gr.Label()

            # The function returns 3 outputs, so we map them here
            btn_live.click(process_call, inputs=[live_audio, live_text], outputs=[out_rep, out_chart, live_text])

        # TAB 2: SIMULATOR
        with gr.TabItem("Call Simulator"):
            btn_sim = gr.Button("📞 Next Incoming Call", variant="primary")
            with gr.Row():
                sim_audio = gr.Audio(label="Stream", interactive=False)
                sim_text = gr.Textbox(label="Transcript", interactive=False)
            with gr.Row():
                sim_rep = gr.Markdown()
                sim_chart = gr.Label()

            btn_sim.click(simulate_call, outputs=[sim_audio, sim_text, sim_rep, sim_chart])

print("🚀 Launching...")
demo.launch(share=True, debug=True)